# CGM-independent HbA1c reanalysis

**Run the numbered cells in order.** This is a new analysis, not a patch to the old CGM-selected table.

The original test results have already been examined. The revised validation/test results will be labeled **post-test revision / exploratory internal evaluation**. Original outputs and models are preserved.

The module files in this folder contain the full implementation. No clinical data are bundled or uploaded. Read `README.md` for the complete variable and quality definitions.


## 1. Locate the code and check the input files

Keep the notebook and both `.py` modules in the same folder. If the automatic folder choices below do not match your extraction location, change `CODE_DIR` to that exact folder. Do not point it at the old package.

Dependencies are listed in `requirements.txt`. If an import fails, install that file into the current notebook kernel using `%pip install -r "FULL_PATH_TO_requirements.txt"`, restart the kernel, and rerun this cell. This workflow does not require `shap` or `jinja2`.


In [ ]:
from pathlib import Path
import sys
import json

CODE_DIR = Path.cwd()
if not (CODE_DIR / "aireadi_no_cgm.py").is_file():
    CODE_DIR = Path(r"D:\POSTDOC\YSN\AimAhead\AI_READI_No_CGM_Reanalysis")
for name in ["aireadi_no_cgm.py", "aireadi_no_cgm_models.py"]:
    if not (CODE_DIR / name).is_file():
        raise FileNotFoundError(f"Set CODE_DIR to the extracted folder containing {name}. Current choice: {CODE_DIR}")
sys.path.insert(0, str(CODE_DIR))

import pandas as pd
from IPython.display import display, Image
from aireadi_no_cgm import Config, new_run, source_paths, prepare
from aireadi_no_cgm_models import (
    selection_reports, train_revision, evaluate_revision, explain_revision,
    load_prepared, verify_lock, additional_quality_sensitivity,
)

CFG = Config(
    dataset=r"D:\POSTDOC\YSN\AimAhead\DATA\aireadi\11a60cdd-6588-4f77-80d8-4e2718613172\dataset",
    analysis_base=r"D:\POSTDOC\YSN\AimAhead\AI_READI_Analysis",
    history_days=7,
    start_search_days=10,
    min_observed_minutes_per_day=600,
    oxygen_min_observed_minutes_per_day=60,
    min_good_days=5,
    min_day_span=5,
    outer_folds=5,
    inner_folds=3,
    bootstrap_draws=2000,
    software_test_only=False,
)
checks = pd.DataFrame([
    {"input": name, "exists": path.is_file(), "path": str(path)}
    for name, path in source_paths(CFG).items()
])
display(checks)
if not checks["exists"].all():
    raise FileNotFoundError("Restore or correct the missing source paths above before continuing.")
print("No CGM files, old CGM-aligned caches, old selected windows, or old frozen models are required.")


## 2. Create a separate run and save the revised rules

The first usable wearable record in `[visit calendar day, visit + 10 days)` anchors seven elapsed 24-hour blocks. No window is chosen using HbA1c values or CGM. UTC midnight represents the visit calendar date, not a known visit clock time. This is retrospective estimation of recorded HbA1c.

The existing source-specific wearable coverage criteria are retained. Stored education coding remains a documented issue for source-codebook review; it is not relabeled as verified years.


In [ ]:
RUN_DIR = new_run(CFG)
print("NEW ANALYSIS FOLDER:", RUN_DIR)
print("Original analysis files remain in their original folders.")
print("Analysis status:", CFG.analysis_status)


## 3. Rebuild predictors and eligibility from the full roster

This reads the original Garmin JSON files. Expect more time than the old cached-table workflow. Every roster participant is preserved in a status table. A common modeling participant needs visit-day laboratory HbA1c and at least one quality-passing wearable source. Missing diet or SDH does not exclude anyone. Missing CGM is never checked.

Schema/unit conflicts stop for source review; they are not silently converted into participant exclusions. `preparation_complete.json` is written only after preparation succeeds.


In [ ]:
analysis_status = prepare(CFG, RUN_DIR)
print("Preparation completed:", RUN_DIR / "preparation_complete.json")


## 4. Inspect cohort flow, missingness, and selection

Review source timing and coverage before modeling. The flow diagram contains actual computed counts. The included/excluded comparisons cover the full roster and the visit-day-HbA1c population separately. No statistical significance threshold selects features or people.


In [ ]:
selection = selection_reports(analysis_status, RUN_DIR)
display(pd.read_csv(RUN_DIR / "cohort_flow.csv"))
display(pd.read_csv(RUN_DIR / "exclusion_reasons.csv"))
missingness = pd.read_csv(RUN_DIR / "selection_and_dictionary" / "missingness_by_selection_and_split.csv")
display(missingness.loc[missingness["status"].eq("Included")])
display(Image(filename=str(RUN_DIR / "selection_and_dictionary" / "cohort_flow.png")))
print("Review source timing:", RUN_DIR / "static_predictor_timing_LOCAL_ONLY.csv")
print("Review wearable quality:", RUN_DIR / "wearable_source_status_LOCAL_ONLY.csv")
print("New eligible participants by split:")
display(analysis_status.groupby("recommended_split")["eligible_common"].sum().rename("n_eligible"))


## 5. Refit and freeze the revised training models

Five outer folds, three inner folds, all eight predictor groups. Median imputation, encoding, scaling, tuning, and ensemble-weight fitting occur within their training partitions. This step can take time. It saves checkpoints and fits final models using only the new training cohort. It never imports the old frozen models.

A new lock records the revised protocol, inputs, prepared rows, code, and fitted models. The lock does not erase prior test exposure.


In [ ]:
oof_predictions = train_revision(CFG, analysis_status, RUN_DIR)
training_performance = pd.read_csv(RUN_DIR / "training" / "oof_reports" / "performance_with_conditional_CI.csv")
display(training_performance.loc[
    training_performance["model"].str.endswith("__ensemble") & training_performance["metric"].eq("MAE")
])
display(pd.read_csv(RUN_DIR / "training" / "final_ensemble_weights.csv"))
print("Revised lock:", RUN_DIR / "revised_frozen_models" / "revision_lock.json")


## 6. Apply the revised frozen models to validation

No refitting or weight updates. Since earlier project validation results informed development, report this as exploratory development evidence. Intervals condition on the fitted models/saved predictions.


In [ ]:
validation_predictions = evaluate_revision(CFG, analysis_status, RUN_DIR, "validation")
validation_reports = RUN_DIR / "validation_exploratory_evaluation" / "reports"
display(pd.read_csv(validation_reports / "performance_with_conditional_CI.csv"))
display(pd.read_csv(validation_reports / "paired_comparisons.csv").query("role == 'designated_primary_exploratory'"))


## 7. Apply the same revised models to the original test split

These participants remain excluded from fitting. Nevertheless, the **original test was examined before this revision**, so the new result is exploratory internal evaluation, not untouched confirmation or external validation. Do not tune models using these results.


In [ ]:
test_predictions = evaluate_revision(CFG, analysis_status, RUN_DIR, "test")
test_reports = RUN_DIR / "test_exploratory_evaluation" / "reports"
display(pd.read_csv(test_reports / "performance_with_conditional_CI.csv"))
paired = pd.read_csv(test_reports / "paired_comparisons.csv")
display(paired.loc[paired["metric"].eq("MAE")])
display(pd.read_csv(test_reports / "calibration_diagnostics.csv"))
display(Image(filename=str(test_reports / "CWDS_calibration.png")))


## 8. Explain every eligible validation participant and create labeled figures

This estimates interventional Shapley values for the frozen full weighted ensemble using a training-only background. It explains **every eligible validation participant**; `background_n=32` is a training reference sample, not a limit on the explained participants. The plot contains no participant-count subtitle. Metadata retains counts and additivity checks.

These are Monte Carlo approximations of Shapley values. Nominal variables appear gray rather than using a fictitious low/high response order. Attribution describes model behavior; it does not establish causality or clinical usefulness. No model training occurs here. `README.md` explains approximation and correlation limitations.


In [ ]:
importance = explain_revision(
    CFG, analysis_status, RUN_DIR,
    split="validation",
    group="CWDS",
    background_n=32,
    n_permutations=2,
)
display(importance.drop(columns="feature"))
SHAP_DIR = RUN_DIR / "validation_CWDS_SHAP"
display(Image(filename=str(SHAP_DIR / "SHAP_beeswarm_and_importance.png")))
print("Vector figures:", SHAP_DIR / "SHAP_beeswarm_and_importance.pdf")
print("Formatted table:", SHAP_DIR / "SHAP_importance.html")


## 9. Locate the manuscript tables and figures

Publication outputs do not replace the source/data-quality audit. Report the new real counts and estimates from this run, and explain why the analysis was revised. Retain and disclose the old analysis rather than presenting this as the original frozen protocol.


In [ ]:
outputs = {
    "Cohort flow": RUN_DIR / "cohort_flow.csv",
    "Selection comparison": RUN_DIR / "selection_and_dictionary" / "included_versus_excluded.csv",
    "Missingness": RUN_DIR / "selection_and_dictionary" / "missingness_by_selection_and_split.csv",
    "Feature dictionary": RUN_DIR / "selection_and_dictionary" / "feature_dictionary.csv",
    "Nested CV": RUN_DIR / "training" / "oof_reports" / "performance_with_conditional_CI.csv",
    "Validation": RUN_DIR / "validation_exploratory_evaluation" / "reports" / "performance_with_conditional_CI.csv",
    "Test (exploratory revision)": RUN_DIR / "test_exploratory_evaluation" / "reports" / "performance_with_conditional_CI.csv",
    "Paired and individual-learner comparisons": RUN_DIR / "test_exploratory_evaluation" / "reports" / "paired_comparisons.csv",
    "SHAP table": RUN_DIR / "validation_CWDS_SHAP" / "SHAP_importance.csv",
}
display(pd.DataFrame([{"output": k, "saved": v.is_file(), "path": str(v)} for k, v in outputs.items()]))
print("All outputs:", RUN_DIR)
print("Keep participant-level files marked LOCAL_ONLY in your approved local environment.")


## 10. Optional: one additional quality sensitivity

This cell is off by default because it is an additional experiment. If run, it relaxes scalar coverage from at least five days spanning five day-index units to at least four days spanning three; all other preparation rules stay the same. It refits training models and provides overlap comparisons to distinguish model/cohort changes. Label it **additional analysis after test evaluation**, not previously prespecified. Do not choose a rule because it yields a more favorable result.


In [ ]:
RUN_ADDITIONAL_SENSITIVITY = False
if RUN_ADDITIONAL_SENSITIVITY:
    SENSITIVITY_DIR = additional_quality_sensitivity(CFG, RUN_DIR)
    print("Additional post-test sensitivity:", SENSITIVITY_DIR)
else:
    print("Additional sensitivity not run.")


## Resume after a kernel restart

Run section 1 to import the code, then set `RUN_DIR` to your **existing revised run**. Do not rerun section 2, preparation, training, or completed evaluation into that folder. The example below is commented out to prevent an accidental reset during Run All. Remove the comments and insert your actual run path when needed.


In [ ]:
# RUN_DIR = Path(r"D:\POSTDOC\YSN\AimAhead\AI_READI_Analysis\hba1c_no_cgm_v1\YOUR_ACTUAL_RUN_FOLDER")
# protocol = json.loads((RUN_DIR / "protocol.json").read_text(encoding="utf-8"))
# CFG = Config(**{key: protocol[key] for key in Config.__dataclass_fields__})
# analysis_status = load_prepared(RUN_DIR)
# if (RUN_DIR / "revised_frozen_models" / "revision_lock.json").is_file():
#     verify_lock(RUN_DIR)
# print("Resumed:", RUN_DIR)
